# 第25讲：llama3_supp_data_config

按原单元编号阅读。配套讲解见本讲的解释.md，运行方法与记录见复现目录。


### 原单元 0（markdown）

补充数据与配置，和主notebook配套；不把第29讲7B/8B逐层对比notebook混入本讲。


## data

### 原单元 1（markdown）

15T+预训练token、10M+人工标注与不含Meta用户数据均为2024官方model card声明；没有下载训练语料或独立审计。token不是样本或字数，公开可获取不等于全部无授权条件。


- Llama 3 was **pretrained** on over 15 trillion tokens（15T） of data from **publicly available sources**.
- The **fine-tuning** data includes publicly available **instruction datasets**, as well as over **10M human-annotated examples**.
- Neither the pretraining nor the fine-tuning datasets include Meta user data.

### 原单元 2（markdown）

进入配置：同一个概念在Meta/HF有不同字段名。


## config

### 原单元 3（markdown）

dim→hidden_size4096，n_layers→num_hidden_layers32，n_heads→num_attention_heads32，n_kv_heads→num_key_value_heads8，vocab_size128256；multiple_of1024与ffn_dim_multiplier1.3共同决定中间宽14336；norm_eps1e-5，rope_theta500000。此JSON没写上下文长度，不能据此猜8k；8k来自官方卡。

新增源码注释（保留上方原片段）：

```jsonc
{
   // 隐藏宽4096，HF对应hidden_size。
   "dim": 4096,
    // 32个TransformerBlock。
    "n_layers": 32,
    // 32个Q头。
    "n_heads": 32,
    // 8个KV头，每组供4个Q头使用。
    "n_kv_heads": 8,
    // 128256，已经含256特殊token。
    "vocab_size": 128256,
    // 中间宽向1024倍数上取整。
    "multiple_of": 1024,
    // 原宽度缩到2/3后再乘1.3。
    "ffn_dim_multiplier": 1.3,
    // RMSNorm稳定项1e-5。
    "norm_eps": 1e-05,
    // RoPE频率底数500000。
    "rope_theta": 500000.0
}
```


- Meta-Llama-3-8B

```
{
   "dim": 4096,
    "n_layers": 32,
    "n_heads": 32,
    "n_kv_heads": 8,
    "vocab_size": 128256,
    "multiple_of": 1024,
    "ffn_dim_multiplier": 1.3,
    "norm_eps": 1e-05,
    "rope_theta": 500000.0
}
```

- rope_theta: base
    - $5\times 10^{5}$

### 原单元 4（markdown）

进入SwiGLU中间宽度推导，4D只是构造参数的初值。


#### ffn layer shape

### 原单元 5（markdown）

FeedForward构造接受dim=D与初始hidden_dim=4D；实际内部先乘2/3再乘1.3并向1024倍数取整。self.feed_forward注册模块；args必须先由ModelArgs构造，片段并非独立可执行程序。

新增源码注释（保留上方原片段）：

```python
# FeedForward构造接受dim=D与初始hidden_dim=4D；实际内部先乘2/3再乘1.3并向1024倍数取整。self.feed_forward注册模块；args必须先由ModelArgs构造，片段并非独立可执行程序。
self.feed_forward = FeedForward(
    dim=args.dim,
    hidden_dim=4 * args.dim,
    multiple_of=args.multiple_of,
    ffn_dim_multiplier=args.ffn_dim_multiplier,
)
```


```
self.feed_forward = FeedForward(
    dim=args.dim,
    hidden_dim=4 * args.dim,
    multiple_of=args.multiple_of,
    ffn_dim_multiplier=args.ffn_dim_multiplier,
)
```

### 原单元 6（code）

hidden_dim先是4×4096=16384，尚未是权重真实宽度；赋值改变Python变量，不分配权重。


In [5]:
hidden_dim = 4*4096

### 原单元 7（markdown）

完整原FeedForward类：继承nn.Module，super初始化注册表。w1/w3是列并行[D→M]，w2行并行[M→D]；SwiGLU为w2(silu(w1(x))*w3(x))，*为逐元素门控。init_method=lambda x:x不初始化数值，本次小实验显式初始化，真实权重通常随后从checkpoint加载。

新增源码注释（保留上方原片段）：

```python
# 完整原FeedForward类：继承nn.Module，super初始化注册表。w1/w3是列并行[D→M]，w2行并行[M→D]；SwiGLU为w2(silu(w1(x))*w3(x))，*为逐元素门控。init_method=lambda x:x不初始化数值，本次小实验显式初始化，真实权重通常随后从checkpoint加载。
# 完整官方SwiGLU层，仅加注释，依赖真实FairScale初始化。
class FeedForward(nn.Module):
    # 实例初始化；self是该对象，不是输入张量。
    def __init__(
        self,
        # 模型隐藏宽度D，输入嵌入/残差都保留此维度。
        dim: int,
        hidden_dim: int,
        # SwiGLU中间宽上取整倍数；8B JSON实际覆盖为1024。
        multiple_of: int,
        # SwiGLU宽度额外倍率，8B实际为1.3，None则不乘。
        ffn_dim_multiplier: Optional[float],
    ):
        # 初始化nn.Module注册机制，随后赋值的子层才可被遍历。
        super().__init__()
        # 先乘2/3并向零截断到10922。
        hidden_dim = int(2 * hidden_dim / 3)
        # custom dim factor multiplier
        # 可选倍率不为None时才应用。
        if ffn_dim_multiplier is not None:
            # 乘配置倍率1.3后再次截断。
            hidden_dim = int(ffn_dim_multiplier * hidden_dim)
        # 将中间宽上取整为multiple_of的整数倍。
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)

        # 门控分支权重[M,D]；world1输出[B,T,M]。
        self.w1 = ColumnParallelLinear(
            dim, hidden_dim, bias=False, gather_output=False, init_method=lambda x: x
        )
        # 输出投影权重[D,M]；world1恢复[B,T,D]。
        self.w2 = RowParallelLinear(
            hidden_dim, dim, bias=False, input_is_parallel=True, init_method=lambda x: x
        )
        # 内容分支权重[M,D]，独立于w1。
        self.w3 = ColumnParallelLinear(
            dim, hidden_dim, bias=False, gather_output=False, init_method=lambda x: x
        )

    # nn.Module.__call__会调用forward；x为浮点[B,T,D]。
    def forward(self, x):
        # silu门控逐元素乘内容，最后投影回D，无bias。
        return self.w2(F.silu(self.w1(x)) * self.w3(x))
```


```
class FeedForward(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        multiple_of: int,
        ffn_dim_multiplier: Optional[float],
    ):
        super().__init__()
        hidden_dim = int(2 * hidden_dim / 3)
        # custom dim factor multiplier
        if ffn_dim_multiplier is not None:
            hidden_dim = int(ffn_dim_multiplier * hidden_dim)
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)

        self.w1 = ColumnParallelLinear(
            dim, hidden_dim, bias=False, gather_output=False, init_method=lambda x: x
        )
        self.w2 = RowParallelLinear(
            hidden_dim, dim, bias=False, input_is_parallel=True, init_method=lambda x: x
        )
        self.w3 = ColumnParallelLinear(
            dim, hidden_dim, bias=False, gather_output=False, init_method=lambda x: x
        )

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))
```

### 原单元 8（code）

依次int(32768/3)=10922，int(10922×1.3)=14198，向1024整数倍上取整到14336。//为整除；必须按原顺序截断，不能先把公式随意约分。


In [6]:
hidden_dim = int(2*hidden_dim / 3)
hidden_dim = int(1.3 * hidden_dim)
hidden_dim = 1024 * ((hidden_dim + 1024 - 1) // 1024)
hidden_dim

14336

### 原单元 9（code）

14336/4096=3.5是最终MLP宽度倍率；三矩阵参数3DM=176160768每层，不是两层4D参数。


In [7]:
14336 / 4096

3.5

### 原单元 10（code）

原空代码单元，保留；新增注释只说明它不执行任何模型工作。
